# 8. Async Parallel Execution

MARES uses Python's `asyncio` to run independent sub-tasks concurrently. The **ParallelRunner** manages concurrency with a configurable cap.

This maximizes throughput while respecting LLM API rate limits.

In [ ]:
import asyncio
import time

from mares.orchestrator.parallel_runner import ParallelRunner
from mares.models.sub_task import SubTask
from mares.models.agent_output import AgentOutput

runner = ParallelRunner(max_concurrency=3)
print(f"Max concurrency: {runner.max_concurrency}")

## Simulating Parallel Work

Let's simulate sub-tasks that take varying amounts of time to see how parallel execution works:

In [ ]:
import asyncio

async def simulate_work(sub_task: SubTask) -> AgentOutput:
    delay = {1: 0.3, 2: 0.5, 3: 0.2, 4: 0.4, 5: 0.1}.get(sub_task.id, 0.2)
    await asyncio.sleep(delay)
    return AgentOutput(
        agent="research_agent",
        sub_task_id=sub_task.id,
        content={"summary": f"Result for task {sub_task.id}"}
    )

# Run a wave in parallel
sub_tasks = [
    SubTask(id=i, task=f"Task {i}", depends_on=[])
    for i in range(1, 6)
]

start = time.perf_counter()
results = await runner.run_wave(sub_tasks, simulate_work)
elapsed = time.perf_counter() - start

print(f"Ran {len(results)} tasks in parallel in {elapsed:.2f}s")
print(f"(Sequential would take ~{0.3+0.5+0.2+0.4+0.1:.1f}s)")

## Gather with Concurrency

The `gather_with_concurrency` utility limits how many coroutines run at once:

In [ ]:
from mares.utils.async_utils import gather_with_concurrency

async def slow_task(n: int) -> str:
    await asyncio.sleep(0.1)
    return f"Task {n} done"

# Run 20 tasks with max 5 concurrent
start = time.perf_counter()
results = await gather_with_concurrency(5, *(slow_task(i) for i in range(20)))
elapsed = time.perf_counter() - start

print(f"20 tasks at concurrency 5: {elapsed:.2f}s (expected ~0.4s)")
print(f"Results: {len(results)}")